# N1 — Fundamentos Conceituais de Agentes Inteligentes e Agentes com LLMs

* Professor: Júlio Cesar dos Reis <a href="mailto:dosreis@unicamp.br">(dosreis@unicamp.br)</a>
* Monitor: Renan dos Santos Morais <a href="mailto:r299211@dac.unicamp.br">(r299211@dac.unicamp.br)</a> 

Este notebook introduz o paradigma de **agentes inteligentes** em alto nível. A proposta é construir a base conceitual necessária antes de entrar em LangGraph, LangChain, tools, estados implementados, nós, transições e  diferentes tipos de reasoning em agentes.

Neste notebook, vamos pensar em agentes como sistemas capazes de:

- perceber informações do ambiente;
- decidir com base em objetivos, contexto e estado;
- agir por meio de respostas, funções, ferramentas ou interações externas;
- repetir esse ciclo até atingir um objetivo ou uma condição de parada.

O contexto usado ao longo dos exemplos será um **curso de LangGraph e agentes com LLMs**. Assim, os conceitos já se conectam naturalmente com os próximos notebooks do curso.


## 0.1 Pré-requisitos

Para acompanhar este notebook, é útil conhecer:

- Python básico;
- noções gerais de IA e modelos de linguagem;
- ideia geral de entrada, processamento e saída em programas;
- nenhuma familiaridade prévia com LangGraph é necessária.

Este notebook é propositalmente **conceitual**.

Não vamos implementar grafos, tools reais, chamadas de LLM ou agentes ReAct ainda.


## 0.2 Objetivos

Ao final deste notebook, você deverá saber:

1. Explicar o que é um agente inteligente.
2. Diferenciar programas tradicionais, chatbots e agentes autônomos.
3. Identificar os elementos fundamentais de um agente: percepção, decisão, ação, estado, memória e objetivo.
4. Descrever diferentes tipos de agentes.
5. Entender o papel dos LLMs em arquiteturas de agentes.
6. Entender prompt como uma primeira camada de especificação comportamental.
7. Diferenciar prompt, estado, memória, planejamento e ferramentas.
8. Especificar um agente antes de implementá-lo.
9. Criar uma ficha conceitual de agente.
10. Entender como esses conceitos serão implementados nos próximos notebooks.


## 0.3 Organização do Notebook

Este notebook está organizado em três blocos conceituais:

### A1 — Fundamentos de agentes inteligentes

1. De programas para agentes.
2. O ciclo percepção → decisão → ação.
3. Diferenças entre programa tradicional, chatbot e agente.
4. Tipos de agentes.
5. Evolução do conceito de agente na IA.

### A2 — Agentes baseados em LLMs

6. O papel do modelo de linguagem.
7. Componentes de um agente com LLM.
8. Prompt como especificação inicial.
9. Few-shot, raciocínio estruturado e critérios de decisão.
10. Ciclo operacional de um agente com LLM.

### A3 — Especificação de agentes

11. Como definir objetivos, entradas, saídas e responsabilidades.
12. Como representar estado e comportamento em alto nível.
13. Como transformar uma ideia de agente em uma especificação.
14. Como essa especificação prepara LangChain, LangGraph e tools.


## 0.4 Contexto usado nos exemplos

Usaremos como exemplo recorrente um possível agente chamado **TutorLangGraph**.

A ideia desse agente é ajudar alunos em um curso sobre agentes com LLMs e LangGraph.

Ele poderia, futuramente:

- responder dúvidas conceituais;
- identificar o módulo relacionado à dúvida;
- sugerir revisão de módulos anteriores;
- consultar progresso do aluno;
- recomendar exercícios;
- montar planos de estudo;
- acionar ferramentas externas.

Neste notebook, porém, tudo isso será tratado apenas em nível conceitual.


## 0.5 Preparação do ambiente

Como este notebook é conceitual, não precisamos instalar bibliotecas externas.

Usaremos apenas Python básico para representar ideias como dicionários, listas e funções simples.


In [25]:
from pprint import pprint
from dataclasses import dataclass, field
from typing import List, Dict, Callable, Optional


# A1 — Fundamentos de agentes inteligentes

Neste primeiro bloco, vamos construir a ideia central de agente.

A pergunta principal é:

> O que diferencia um agente de um programa tradicional ou de um chatbot simples?


# 1. De programas para agentes

Um programa tradicional normalmente executa um fluxo previamente definido.

Por exemplo:

1. recebe uma entrada;
2. aplica uma regra;
3. devolve uma saída.

Um agente, por outro lado, é pensado como um sistema que opera em um ambiente. Ele recebe percepções, mantém contexto, toma decisões e executa ações para alcançar algum objetivo.

Uma forma simples de resumir é:

```text
Programa tradicional: entrada → processamento → saída

Agente: percepção → estado → decisão → ação → nova percepção
```

A diferença importante não é apenas técnica. É arquitetural.

Um agente precisa ser pensado em termos de **objetivo**, **ambiente**, **estado**, **decisões** e **ações possíveis**.


In [26]:
programa_tradicional = {
    "entrada": "nota_final",
    "regra": "se nota_final >= 7, aluno aprovado",
    "saida": "aprovado ou reprovado"
}

agente_tutor = {
    "ambiente": "curso de LangGraph",
    "percepcoes": ["mensagem do aluno", "módulo atual", "histórico de dúvidas"],
    "objetivo": "ajudar o aluno a avançar no curso",
    "estado": ["aluno", "módulo atual", "dúvida atual", "histórico recente"],
    "acoes": ["explicar", "perguntar", "recomendar revisão", "sugerir exercício"]
}

pprint(programa_tradicional)
print()
pprint(agente_tutor)


{'entrada': 'nota_final',
 'regra': 'se nota_final >= 7, aluno aprovado',
 'saida': 'aprovado ou reprovado'}

{'acoes': ['explicar', 'perguntar', 'recomendar revisão', 'sugerir exercício'],
 'ambiente': 'curso de LangGraph',
 'estado': ['aluno', 'módulo atual', 'dúvida atual', 'histórico recente'],
 'objetivo': 'ajudar o aluno a avançar no curso',
 'percepcoes': ['mensagem do aluno', 'módulo atual', 'histórico de dúvidas']}


## Discussão

No exemplo anterior, o programa tradicional resolve uma tarefa fechada.

O agente, por outro lado, precisa lidar com uma situação mais aberta:

- o aluno pode ter diferentes dúvidas;
- a dúvida pode depender de módulos anteriores;
- a resposta pode exigir explicação, revisão ou sugestão de exercício;
- o agente precisa decidir qual ação faz mais sentido.

Isso já introduz uma diferença fundamental:

> Um agente não é definido apenas pelo que ele responde, mas pelo modo como percebe, decide e age.


# 2. O que é um agente inteligente?

Um **agente inteligente** pode ser entendido como um sistema que:

1. percebe informações do ambiente;
2. interpreta essas informações com base em algum estado ou conhecimento;
3. faz reasoning ('thinking');
3. decide o que fazer;
4. executa uma ação;
5. avalia novas informações após agir.

Esse ciclo costuma ser representado assim:

```text
Ambiente → Percepção → Estado → Decisão → Ação → Ambiente
```

Esse modelo é geral. Ele pode descrever desde agentes robóticos até agentes baseados em modelos de linguagem.


In [27]:
ciclo_agente = [
    "perceber o ambiente",
    "atualizar ou consultar estado",
    "decidir a próxima ação",
    "executar a ação",
    "observar o resultado",
    "continuar ou encerrar"
]

for passo, descricao in enumerate(ciclo_agente, start=1):
    print(f"{passo}. {descricao}")


1. perceber o ambiente
2. atualizar ou consultar estado
3. decidir a próxima ação
4. executar a ação
5. observar o resultado
6. continuar ou encerrar


## 2.1 Percepção, decisão e ação

Os três elementos mais básicos de um agente são:

| Elemento | Pergunta central | Exemplo no curso |
|---|---|---|
| Percepção | O que o agente observa? | Mensagem do aluno |
| Reasoning | O que o agente 'pensa'? | Analisa a mensagem do aluno |
| Decisão | O que o agente escolhe fazer? | Explicar, perguntar ou recomendar revisão |
| Ação | Como o agente interfere no ambiente? | Enviar resposta, sugerir exercício, chamar ferramenta |

Esses três elementos formam o núcleo do comportamento agêntico.


In [28]:
percepcao = "Não entendi o que é estado em um agente."

possiveis_decisoes = [
    "responder diretamente",
    "perguntar qual módulo o aluno está estudando",
    "recomendar revisão de conceitos anteriores",
    "mostrar um exemplo conceitual"
]

acao_escolhida = "mostrar um exemplo conceitual"

print("Percepção:", percepcao)
print("Decisões possíveis:")
for decisao in possiveis_decisoes:
    print("-", decisao)
print("Ação escolhida:", acao_escolhida)


Percepção: Não entendi o que é estado em um agente.
Decisões possíveis:
- responder diretamente
- perguntar qual módulo o aluno está estudando
- recomendar revisão de conceitos anteriores
- mostrar um exemplo conceitual
Ação escolhida: mostrar um exemplo conceitual


## 2.2 Exemplo: ciclo de um tutor conceitual

Imagine o seguinte cenário:

```text
Aluno: "Estou no A5, mas não entendi por que um agente precisa de estado."
```

Um agente tutor poderia operar assim:

```text
Percepção:
- O aluno está no A5.

Reasoning:
- A dúvida é sobre estado.
- Estado é um conceito necessário antes de workflows completos.

Decisão:
- Revisar o conceito de estado antes de explicar A5.

Ação:
- Explicar estado com exemplo simples e relacionar com o curso.
```

Observe que o agente não apenas responde. Ele interpreta o contexto e escolhe uma estratégia de resposta.


In [29]:
def politica_tutor(mensagem: str, modulo_atual: str) -> str:
    """Política conceitual simples, sem LLM e sem framework."""
    mensagem_lower = mensagem.lower()

    if "estado" in mensagem_lower:
        return "revisar conceito de estado antes de avançar"
    if "tool" in mensagem_lower or "ferramenta" in mensagem_lower:
        return "explicar que ferramentas ampliam ações do agente"
    if modulo_atual == "A6":
        return "verificar se A4 e A5 estão consolidados"
    return "responder com explicação conceitual"

politica_tutor("Estou no A5 e não entendi estado", "A5")


'revisar conceito de estado antes de avançar'

## Discussão

A função anterior não é um agente completo. Ela é apenas uma simulação simples de uma política de decisão.

Mesmo assim, ela ajuda a separar três elementos:

- entrada percebida;
- regra ou estratégia de decisão;
- a tomada de decisão;
- ação sugerida.

Em agentes com LLMs, essa decisão pode ser mais flexível, mas a estrutura conceitual continua semelhante.


# 3. Programa tradicional, chatbot e agente

É útil diferenciar três níveis:

| Sistema | Característica principal | Exemplo |
|---|---|---|
| Programa tradicional | segue fluxo fixo | calcular média do aluno |
| Chatbot simples | responde mensagens | explicar o que é LangGraph |
| Agente | decide ações em função de objetivo, estado e ambiente | consultar progresso, decidir revisão e montar plano |

Um chatbot pode ser parte de um agente, mas nem todo chatbot é um agente.

Um agente costuma envolver uma arquitetura mais ampla ao redor do modelo ou da interface textual.


In [30]:
def programa_calcula_media(notas: List[float]) -> str:
    media = sum(notas) / len(notas)
    return "aprovado" if media >= 7 else "revisar conteúdo"

chatbot_resposta = "LangGraph é um framework para criar agentes e workflows baseados em grafos."

agente_decisao = {
    "percepcao": "aluno com média baixa no módulo A4",
    "estado": {"modulo": "A4", "media": 5.8, "tentativas": 2},
    "decisao": "recomendar revisão antes de avançar para A5",
    "acao": "montar plano de recuperação conceitual"
}

print("Programa:", programa_calcula_media([6, 7, 8]))
print("Chatbot:", chatbot_resposta)
print("Agente:")
pprint(agente_decisao)


Programa: aprovado
Chatbot: LangGraph é um framework para criar agentes e workflows baseados em grafos.
Agente:
{'acao': 'montar plano de recuperação conceitual',
 'decisao': 'recomendar revisão antes de avançar para A5',
 'estado': {'media': 5.8, 'modulo': 'A4', 'tentativas': 2},
 'percepcao': 'aluno com média baixa no módulo A4'}


## 3.1 A autonomia é sempre limitada

Ao falar de agentes, é comum usar a palavra "autônomo".

Mas, em sistemas reais, autonomia não significa liberdade absoluta.

Um agente bem projetado deve operar dentro de limites:

- quais ações pode executar;
- quais dados pode acessar;
- quando deve pedir confirmação humana;
- quais decisões não pode tomar sozinho;
- quando deve parar.

Portanto, ao projetar um agente, também projetamos seus limites.


# 4. Evolução do conceito de agente na IA

A ideia de agente não surgiu com LLMs.

Ela aparece em diferentes momentos da IA:

| Fase | Ideia dominante | Relação com agentes |
|---|---|---|
| Sistemas baseados em regras | decisões explícitas | agente segue regras codificadas |
| Planejamento simbólico | busca por sequência de ações | agente planeja para atingir objetivo |
| Aprendizado de máquina | inferência a partir de dados | agente usa modelos para decidir |
| Aprendizado por reforço | ação, recompensa e ambiente | agente aprende políticas de ação |
| LLMs | linguagem, raciocínio e uso de ferramentas | agente interpreta instruções e coordena ações |

A novidade dos LLMs não é criar o conceito de agente, mas tornar mais natural construir agentes que interagem por linguagem.


# 5. Tipos de agentes

Existem muitas formas de classificar agentes. Para este curso, usaremos uma taxonomia prática.

| Tipo de agente | Ideia principal | Exemplo no curso |
|---|---|---|
| Reativo | responde diretamente à percepção atual | responder uma dúvida simples |
| Baseado em estado | considera contexto interno | lembrar que o aluno está no A5 |
| Baseado em objetivo | escolhe ações para atingir uma meta | ajudar o aluno a concluir A6 |
| Baseado em planejamento | decompõe tarefas em etapas | montar plano de estudo |
| Com ferramentas | chama recursos externos | consultar cronograma ou banco de progresso |
| Multiagente | combina agentes especializados | tutor, avaliador e planejador |

Esses tipos não são categorias rígidas. Um mesmo sistema pode combinar vários deles.


In [31]:
tipos_de_agentes = [
    {
        "tipo": "reativo",
        "memoria": "mínima",
        "decisao": "resposta imediata",
        "exemplo": "responder o que é um nó"
    },
    {
        "tipo": "baseado em estado",
        "memoria": "estado da interação atual",
        "decisao": "depende do contexto",
        "exemplo": "saber que a dúvida atual depende do A4"
    },
    {
        "tipo": "com planejamento",
        "memoria": "estado + objetivo",
        "decisao": "decompor em etapas",
        "exemplo": "criar plano de estudo para A5"
    },
    {
        "tipo": "com ferramentas",
        "memoria": "estado + resultados externos",
        "decisao": "decidir quando chamar recursos externos",
        "exemplo": "consultar progresso do aluno"
    }
]

pprint(tipos_de_agentes)


[{'decisao': 'resposta imediata',
  'exemplo': 'responder o que é um nó',
  'memoria': 'mínima',
  'tipo': 'reativo'},
 {'decisao': 'depende do contexto',
  'exemplo': 'saber que a dúvida atual depende do A4',
  'memoria': 'estado da interação atual',
  'tipo': 'baseado em estado'},
 {'decisao': 'decompor em etapas',
  'exemplo': 'criar plano de estudo para A5',
  'memoria': 'estado + objetivo',
  'tipo': 'com planejamento'},
 {'decisao': 'decidir quando chamar recursos externos',
  'exemplo': 'consultar progresso do aluno',
  'memoria': 'estado + resultados externos',
  'tipo': 'com ferramentas'}]


## Exercício 1 — Classificando agentes

Para cada caso abaixo, diga qual tipo de agente parece mais adequado:

1. Um sistema que responde dúvidas frequentes sobre o curso.
2. Um tutor que lembra quais módulos o aluno concluiu.
3. Um sistema que monta plano de estudo de sete dias.
4. Um agente que consulta uma API para verificar materiais disponíveis.
5. Um conjunto de agentes em que um explica, outro avalia e outro revisa.

Não há necessariamente uma única resposta correta. O objetivo é justificar a classificação.


# A2 — Agentes baseados em LLMs

Neste bloco, vamos introduzir agentes que usam modelos de linguagem.

A ideia principal é:

> A LLM é uma parte do agente, não o agente inteiro.

Um agente com LLM normalmente combina modelo, instruções, estado, memória, planejamento, ferramentas, políticas de controle e critérios de parada.


# 6. O que muda com LLMs?

Modelos de linguagem ampliam a capacidade de agentes porque conseguem:

- interpretar instruções em linguagem natural;
- gerar respostas contextualizadas;
- classificar intenções;
- transformar pedidos abertos em estruturas mais organizadas;
- decidir entre alternativas descritas em texto;
- produzir planos de alto nível;
- solicitar o uso de ferramentas quando necessário.

Mas é importante manter uma distinção:

```text
LLM: componente de linguagem, interpretação e decisão.
Agente: arquitetura completa que usa a LLM dentro de um ciclo de percepção, decisão e ação.
```


## 6.1 Um chatbot com LLM ainda pode não ser um agente

Um chatbot simples pode receber uma pergunta e gerar uma resposta.

Um agente com LLM deve ter algo a mais, por exemplo:

- objetivo persistente;
- estado da tarefa;
- capacidade de escolher ações;
- ferramentas externas;
- memória;
- etapas de planejamento;
- critérios de parada.

Portanto, o uso de LLM não transforma automaticamente um sistema em agente.


# 7. Arquitetura conceitual de um agente com LLM

Uma arquitetura conceitual de agente com LLM pode ser representada assim:

```text
Entrada do usuário
      ↓
Percepção e interpretação
      ↓
Estado / memória / contexto
      ↓
Modelo de linguagem
      ↓
Decisão: responder, planejar, perguntar ou agir
      ↓
Ação: resposta, ferramenta, consulta ou atualização de estado
      ↓
Nova observação ou resposta final
```

Em alto nível, podemos organizar os componentes da seguinte forma:

```text
Agente com LLM
├── modelo de linguagem
├── prompt / instruções
├── estado
├── memória
├── raciocínio 
├── planejamento
├── ferramentas
├── política de decisão
├── critérios de parada
└── interface de entrada e saída
```


In [32]:
arquitetura_agente_llm = {
    "modelo": "interpreta linguagem, gera respostas e pode apoiar decisões",
    "prompt": "define papel, objetivo, restrições e formato esperado",
    "estado": "guarda informações da execução atual",
    "memoria": "recupera informações de interações ou dados anteriores",
    "raciocinio": "organiza análise, classificação e escolha de ação",
    "planejamento": "decompõe objetivos em etapas",
    "ferramentas": "permitem consultar ou executar recursos externos",
    "criterios_de_parada": "definem quando encerrar o ciclo"
}

pprint(arquitetura_agente_llm)


{'criterios_de_parada': 'definem quando encerrar o ciclo',
 'estado': 'guarda informações da execução atual',
 'ferramentas': 'permitem consultar ou executar recursos externos',
 'memoria': 'recupera informações de interações ou dados anteriores',
 'modelo': 'interpreta linguagem, gera respostas e pode apoiar decisões',
 'planejamento': 'decompõe objetivos em etapas',
 'prompt': 'define papel, objetivo, restrições e formato esperado',
 'raciocinio': 'organiza análise, classificação e escolha de ação'}


## 7.1 Modelo de linguagem

O modelo de linguagem é o componente que interpreta e gera linguagem.

Em um agente, ele pode ser usado para:

- entender a intenção do usuário;
- resumir contexto;
- escolher entre ações possíveis;
- gerar uma resposta final;
- produzir um plano;
- decidir se precisa de uma ferramenta.

Mas o modelo não deve ser confundido com todo o sistema.

A arquitetura ao redor do modelo é o que define o agente.


## 7.2 Prompt ou instruções

O prompt é uma forma inicial de especificar o comportamento esperado do modelo.

Ele pode definir:

- papel;
- objetivo;
- contexto;
- limites;
- estilo de resposta;
- formato de saída;
- critérios para tomada de decisão.

Exemplo:

```text
Você é um tutor de um curso de agentes com LLMs.
Seu objetivo é ajudar alunos a entender conceitos antes de avançar para implementação.
Quando a dúvida depender de conceitos anteriores, recomende revisão.
Não invente dados sobre progresso do aluno.
Responda de forma didática e progressiva.
```

Esse prompt ainda não cria um agente completo. Ele apenas descreve comportamento esperado.


In [33]:
prompt_inicial_tutor = """
Você é o TutorLangGraph, um agente educacional para um curso de agentes com LLMs.

Objetivo:
Ajudar alunos a entenderem conceitos de agentes, LangGraph e uso de ferramentas.

Regras:
- Identifique o módulo relacionado à dúvida.
- Explique primeiro o conceito antes de sugerir implementação.
- Não invente informações sobre o progresso do aluno.
- Quando necessário, sugira revisão de módulos anteriores.

Formato da resposta:
1. Diagnóstico da dúvida
2. Explicação conceitual
3. Relação com o curso
4. Próximo passo recomendado
"""

print(prompt_inicial_tutor)



Você é o TutorLangGraph, um agente educacional para um curso de agentes com LLMs.

Objetivo:
Ajudar alunos a entenderem conceitos de agentes, LangGraph e uso de ferramentas.

Regras:
- Identifique o módulo relacionado à dúvida.
- Explique primeiro o conceito antes de sugerir implementação.
- Não invente informações sobre o progresso do aluno.
- Quando necessário, sugira revisão de módulos anteriores.

Formato da resposta:
1. Diagnóstico da dúvida
2. Explicação conceitual
3. Relação com o curso
4. Próximo passo recomendado



## 7.3 Estado

Estado é a informação que o agente mantém durante uma execução.

Exemplo de estado em uma interação do curso:

```text
aluno: Renan
módulo atual: A5
dúvida atual: estado compartilhado
histórico recente: perguntou antes sobre nós e transições
necessita revisão: sim
```

O estado permite que o agente não trate cada mensagem como se fosse isolada.


In [34]:
estado_interacao = {
    "aluno": "Renan",
    "modulo_atual": "A5",
    "duvida_atual": "estado compartilhado",
    "historico_recente": [
        "perguntou sobre nós",
        "perguntou sobre transições"
    ],
    "necessita_revisao": True
}

pprint(estado_interacao)


{'aluno': 'Renan',
 'duvida_atual': 'estado compartilhado',
 'historico_recente': ['perguntou sobre nós', 'perguntou sobre transições'],
 'modulo_atual': 'A5',
 'necessita_revisao': True}


## 7.4 Memória

Memória é diferente de estado.

Uma distinção prática:

| Conceito | Escopo | Exemplo |
|---|---|---|
| Estado | execução atual | dúvida atual, módulo atual, decisão atual |
| Memória de curto prazo | conversa recente | últimas mensagens do aluno |
| Memória de longo prazo | entre sessões | módulos concluídos, preferências, histórico de desempenho |

Em agentes reais, memória pode ser implementada de várias formas: histórico de mensagens, banco de dados, arquivos, vetores, perfis de usuário ou stores externos.

Neste notebook, basta entender a diferença conceitual.


In [35]:
memoria_curta = [
    "Aluno perguntou o que é estado.",
    "Aluno confundiu estado com memória.",
    "Tutor explicou que estado é da execução atual."
]

memoria_longa = {
    "aluno": "Renan",
    "modulos_concluidos": ["A1", "A2"],
    "dificuldades_recorrentes": ["diferenciar estado e memória"]
}

print("Memória curta:")
pprint(memoria_curta)
print("\nMemória longa:")
pprint(memoria_longa)


Memória curta:
['Aluno perguntou o que é estado.',
 'Aluno confundiu estado com memória.',
 'Tutor explicou que estado é da execução atual.']

Memória longa:
{'aluno': 'Renan',
 'dificuldades_recorrentes': ['diferenciar estado e memória'],
 'modulos_concluidos': ['A1', 'A2']}


## 7.5 Raciocínio

Em agentes, raciocínio significa organizar a interpretação de uma situação para escolher uma ação.

Neste curso, vamos preferir falar em ``**raciocínio estruturado**'' em vez de expor cadeias longas de pensamento.

Por exemplo, em vez de pedir:

```text
Pense passo a passo e mostre todo o raciocínio.
```

É melhor especificar etapas observáveis:

```text
1. Identifique o módulo relacionado.
2. Diga quais conceitos prévios são necessários.
3. Escolha entre explicar, perguntar ou recomendar revisão.
4. Responda de forma objetiva.
```

Isso cria estrutura sem depender da exposição completa do raciocínio interno do modelo.


In [36]:
raciocinio_estruturado = [
    "identificar o módulo relacionado",
    "detectar conceito principal da dúvida",
    "verificar pré-requisitos conceituais",
    "escolher estratégia de resposta",
    "gerar resposta final com próximo passo"
]

for etapa in raciocinio_estruturado:
    print("-", etapa)


- identificar o módulo relacionado
- detectar conceito principal da dúvida
- verificar pré-requisitos conceituais
- escolher estratégia de resposta
- gerar resposta final com próximo passo


## 7.6 Planejamento

Planejamento é a decomposição de um objetivo em etapas.

Exemplo:

```text
Objetivo: ajudar o aluno a entender A5.

Plano:
1. verificar se ele entendeu A4;
2. revisar estado e transições;
3. explicar fluxo condicional;
4. mostrar exemplo simples;
5. propor exercício.
```

Planejamento é especialmente importante quando a tarefa não pode ser resolvida em uma única resposta direta.


In [37]:
objetivo = "ajudar o aluno a entender workflows completos no A5"

plano = [
    "confirmar entendimento de estado",
    "revisar nós e transições",
    "introduzir execução condicional",
    "explicar loops de interação",
    "propor exercício integrador"
]

print("Objetivo:", objetivo)
print("Plano:")
for i, etapa in enumerate(plano, start=1):
    print(f"{i}. {etapa}")


Objetivo: ajudar o aluno a entender workflows completos no A5
Plano:
1. confirmar entendimento de estado
2. revisar nós e transições
3. introduzir execução condicional
4. explicar loops de interação
5. propor exercício integrador


## 7.7 Ferramentas

Ferramentas são recursos externos que ampliam o que o agente pode fazer.

Exemplos:

| Ferramenta | O que permite |
|---|---|
| API | buscar dados externos |
| Banco de dados | consultar progresso ou registros |
| Calculadora | executar cálculo confiável |
| Buscador | recuperar documentos |
| Executor de código | testar uma solução |

Ponto central:

> A LLM pode decidir solicitar uma ferramenta, mas quem executa a ferramenta é o sistema ao redor dela.




In [38]:
ferramentas_futuras = {
    "consultar_cronograma": "retorna informações dos módulos do curso",
    "consultar_progresso": "retorna módulos concluídos por um aluno",
    "gerar_plano_estudo": "monta um plano com base em dificuldade e tempo disponível"
}

pprint(ferramentas_futuras)


{'consultar_cronograma': 'retorna informações dos módulos do curso',
 'consultar_progresso': 'retorna módulos concluídos por um aluno',
 'gerar_plano_estudo': 'monta um plano com base em dificuldade e tempo '
                       'disponível'}


# 8. Prompt como a primeira especificação do agente

Prompt não é o agente inteiro.

Prompt é uma camada de especificação comportamental.

Ele ajuda a responder perguntas como:

- Quem é o agente?
- Qual é seu objetivo?
- Que tipo de resposta deve produzir?
- Quais restrições deve respeitar?
- Como deve decidir entre alternativas?

Mas prompts sozinhos não resolvem tudo. Para agentes robustos, também precisamos de arquitetura: estado, memória, ferramentas, fluxo, monitoramento e critérios de parada.


## 8.1 Elementos essenciais de um prompt de agente

Um prompt de agente pode incluir:

| Elemento | Função |
|---|---|
| Papel | define identidade operacional |
| Objetivo | define a meta principal |
| Contexto | informa domínio de atuação |
| Restrições | define limites |
| Critérios de decisão | orienta escolhas |
| Formato de saída | padroniza resposta |
| Exemplos | mostram comportamento esperado |

Esses elementos funcionam como uma primeira versão de contrato entre projetista e modelo.


In [39]:
def construir_prompt_agente(nome: str, papel: str, objetivo: str, restricoes: List[str]) -> str:
    restricoes_formatadas = "\n".join(f"- {r}" for r in restricoes)
    return f"""
Nome do agente: {nome}
Papel: {papel}
Objetivo: {objetivo}

Restrições:
{restricoes_formatadas}

Responda sempre com:
1. Diagnóstico
2. Explicação
3. Próximo passo
""".strip()

prompt = construir_prompt_agente(
    nome="TutorLangGraph",
    papel="tutor conceitual de um curso de agentes com LLMs",
    objetivo="ajudar alunos a entenderem agentes antes de implementar LangGraph",
    restricoes=[
        "não antecipar código de LangGraph neste módulo",
        "não inventar progresso do aluno",
        "explicar conceitos antes de sugerir implementação"
    ]
)

print(prompt)


Nome do agente: TutorLangGraph
Papel: tutor conceitual de um curso de agentes com LLMs
Objetivo: ajudar alunos a entenderem agentes antes de implementar LangGraph

Restrições:
- não antecipar código de LangGraph neste módulo
- não inventar progresso do aluno
- explicar conceitos antes de sugerir implementação

Responda sempre com:
1. Diagnóstico
2. Explicação
3. Próximo passo


## 8.2 Role prompting

Role prompting é a técnica de definir explicitamente o papel do modelo.

Exemplo:

```text
Você é um tutor de um curso introdutório sobre agentes inteligentes.
```

Isso ajuda a condicionar estilo, escopo e responsabilidade.

No entanto, papel não é suficiente. Um bom agente também precisa de objetivo, limites e critérios de decisão.


## 8.3 Objetivos e restrições

Objetivos dizem o que o agente deve alcançar.

Restrições dizem o que ele não deve fazer.

Exemplo:

```text
Objetivo:
Ajudar o aluno a entender agentes com LLMs.

Restrições:
- Não entrar em LangGraph ainda.
- Não apresentar tools como se fossem executadas pela LLM.
- Não inventar dados sobre o aluno.
```

Em agentes robustos, restrições são tão importantes quanto objetivos.


## 8.4 Formato de saída

Definir formato de saída ajuda a tornar respostas mais consistentes.

Exemplo:

```text
Responda no formato:
1. Conceito principal
2. Exemplo no curso
3. Erro comum
4. Próximo passo
```

Isso é útil quando o agente precisa produzir respostas comparáveis, avaliáveis ou fáceis de processar.


## 8.5 Few-shot prompting

Few-shot prompting significa fornecer exemplos de entrada e saída esperadas.

Ele é útil quando queremos reduzir ambiguidade sobre o comportamento desejado.

Exemplo:

```text
Exemplo 1:
Aluno: "O que é estado?"
Resposta esperada:
- Estado é a informação mantida durante a execução.
- Em agentes, ajuda a decidir com base no contexto.
- No curso, esse conceito será importante antes de LangGraph.

Exemplo 2:
Aluno: "O que é ferramenta?"
Resposta esperada:
- Ferramenta é uma capacidade externa ao modelo.
- O agente pode decidir usá-la.
- A execução é feita pelo sistema, não pela LLM.
```

Few-shot é especialmente útil quando o agente precisa seguir um padrão de atendimento, classificação ou resposta.


In [40]:
few_shot_tutor = [
    {
        "entrada": "Não entendi o que é estado.",
        "saida_esperada": {
            "diagnostico": "dúvida conceitual sobre componente interno de agentes",
            "explicacao": "estado é a informação mantida durante uma execução",
            "proximo_passo": "revisar percepção, decisão e ação"
        }
    },
    {
        "entrada": "Quando um agente usa ferramenta?",
        "saida_esperada": {
            "diagnostico": "dúvida sobre ações externas",
            "explicacao": "ferramentas são usadas quando o agente precisa consultar ou executar algo fora da LLM",
            "proximo_passo": "estudar tools e ReAct no módulo futuro"
        }
    }
]

pprint(few_shot_tutor)


[{'entrada': 'Não entendi o que é estado.',
  'saida_esperada': {'diagnostico': 'dúvida conceitual sobre componente '
                                    'interno de agentes',
                     'explicacao': 'estado é a informação mantida durante uma '
                                   'execução',
                     'proximo_passo': 'revisar percepção, decisão e ação'}},
 {'entrada': 'Quando um agente usa ferramenta?',
  'saida_esperada': {'diagnostico': 'dúvida sobre ações externas',
                     'explicacao': 'ferramentas são usadas quando o agente '
                                   'precisa consultar ou executar algo fora da '
                                   'LLM',
                     'proximo_passo': 'estudar tools e ReAct no módulo '
                                      'futuro'}}]


## 8.6 Chain-of-Thought e raciocínio estruturado

Chain-of-Thought é uma técnica associada à ideia de induzir o modelo a raciocinar em etapas.

Em vez de orientar o aluno a pedir a cadeia completa de pensamento do modelo, vamos preferir **raciocínio estruturado observável**.

Exemplo recomendado:

```text
Antes de responder:
1. Identifique o tópico principal.
2. Identifique conceitos prévios necessários.
3. Escolha se deve explicar, perguntar ou recomendar revisão.
4. Produza uma resposta objetiva.
```

Essa abordagem estrutura o comportamento do agente sem depender de exposição detalhada do raciocínio interno.


## 8.7 Critérios de decisão

Critérios de decisão aproximam o prompt de uma política de ação.

Exemplo:

```text
Se a dúvida for conceitual, explique com exemplo.
Se a dúvida depender de módulo anterior, recomende revisão.
Se faltar informação sobre progresso, pergunte ou consulte dados.
Se a tarefa exigir dado externo, use uma ferramenta futuramente.
```

Esses critérios preparam a transição para agentes que tomam decisões em ciclos.


In [41]:
criterios_decisao = [
    {
        "condicao": "dúvida conceitual",
        "acao": "explicar com analogia e exemplo"
    },
    {
        "condicao": "dúvida depende de módulo anterior",
        "acao": "recomendar revisão antes de avançar"
    },
    {
        "condicao": "falta informação sobre progresso",
        "acao": "pedir informação ou consultar base externa futuramente"
    },
    {
        "condicao": "necessidade de dado externo",
        "acao": "usar ferramenta em módulo futuro"
    }
]

pprint(criterios_decisao)


[{'acao': 'explicar com analogia e exemplo', 'condicao': 'dúvida conceitual'},
 {'acao': 'recomendar revisão antes de avançar',
  'condicao': 'dúvida depende de módulo anterior'},
 {'acao': 'pedir informação ou consultar base externa futuramente',
  'condicao': 'falta informação sobre progresso'},
 {'acao': 'usar ferramenta em módulo futuro',
  'condicao': 'necessidade de dado externo'}]


# 9. Ciclo operacional de um agente com LLM

Mesmo sem implementar uma LLM, podemos descrever o ciclo operacional:

```text
1. Receber entrada do usuário.
2. Interpretar a intenção.
3. Atualizar ou consultar estado.
4. Decidir próxima ação.
5. Responder, perguntar, planejar ou solicitar ferramenta.
6. Observar o resultado.
7. Encerrar ou continuar.
```

Esse ciclo será importante para entender agentes ReAct posteriormente.


In [42]:
def ciclo_conceitual_agente(mensagem: str, estado: Dict) -> Dict:
    """Simulação conceitual de um ciclo de agente, sem LLM."""

    if "estado" in mensagem.lower():
        topico = "estado"
        acao = "explicar conceito"
    elif "ferramenta" in mensagem.lower() or "tool" in mensagem.lower():
        topico = "ferramentas"
        acao = "explicar papel de ferramentas"
    else:
        topico = "geral"
        acao = "responder conceitualmente"

    novo_estado = dict(estado)
    novo_estado["ultima_mensagem"] = mensagem
    novo_estado["topico_detectado"] = topico
    novo_estado["acao_planejada"] = acao

    return novo_estado

estado_inicial = {"aluno": "Renan", "modulo": "A1-A3"}
novo_estado = ciclo_conceitual_agente("O que é estado em um agente?", estado_inicial)
pprint(novo_estado)


{'acao_planejada': 'explicar conceito',
 'aluno': 'Renan',
 'modulo': 'A1-A3',
 'topico_detectado': 'estado',
 'ultima_mensagem': 'O que é estado em um agente?'}


## Discussão

A função anterior não usa LLM.

Ela serve apenas para mostrar que um agente pode ser analisado como um ciclo de atualização de estado e decisão de ação.

Nos próximos notebooks, esse tipo de ciclo será implementado com componentes mais concretos:

- estado explícito;
- nós de processamento;
- transições;
- loops;
- ferramentas;
- mensagens.


# A3 — Especificação de agentes

Neste bloco, vamos transformar os conceitos anteriores em uma prática de projeto.

A ideia principal é:

> Antes de implementar um agente, especifique o agente.

Uma boa especificação evita que o agente seja apenas um prompt solto ou uma coleção desorganizada de funções.


# 10. Como especificar um agente

Uma especificação de agente deve responder perguntas como:

```text
Quem é o agente?
Qual objetivo ele persegue?
O que ele observa?
O que ele sabe?
O que ele pode fazer?
Como ele decide?
Quando ele termina?
Quais limites ele deve respeitar?
```

Essas perguntas ajudam a passar de uma ideia vaga para uma arquitetura projetável.


## 10.1 Ficha de especificação

Uma ficha mínima pode conter:

```text
Nome do agente:
Objetivo principal:
Usuário-alvo:
Entradas esperadas:
Saídas esperadas:
Estado necessário:
Memórias necessárias:
Ferramentas futuras:
Regras de comportamento:
Critérios de sucesso:
Critérios de parada:
Falhas esperadas:
O que o agente não deve fazer:
```

Essa ficha ainda não é código. Ela é uma etapa de projeto.


In [43]:
ficha_vazia = {
    "nome": "",
    "objetivo_principal": "",
    "usuario_alvo": "",
    "entradas_esperadas": [],
    "saidas_esperadas": [],
    "estado_necessario": [],
    "memorias_necessarias": [],
    "ferramentas_futuras": [],
    "regras_de_comportamento": [],
    "criterios_de_sucesso": [],
    "criterios_de_parada": [],
    "falhas_esperadas": [],
    "nao_deve_fazer": []
}

pprint(ficha_vazia)


{'criterios_de_parada': [],
 'criterios_de_sucesso': [],
 'entradas_esperadas': [],
 'estado_necessario': [],
 'falhas_esperadas': [],
 'ferramentas_futuras': [],
 'memorias_necessarias': [],
 'nao_deve_fazer': [],
 'nome': '',
 'objetivo_principal': '',
 'regras_de_comportamento': [],
 'saidas_esperadas': [],
 'usuario_alvo': ''}


# 11. Exemplo de especificação: TutorLangGraph

Agora vamos preencher a ficha para o agente usado como exemplo no curso.


In [44]:
ficha_tutor_langgraph = {
    "nome": "TutorLangGraph",
    "objetivo_principal": "ajudar alunos a entenderem agentes com LLMs e LangGraph de forma progressiva",
    "usuario_alvo": "alunos iniciantes ou intermediários no curso",
    "entradas_esperadas": [
        "dúvidas conceituais",
        "pedidos de exemplo",
        "dúvidas sobre módulos do curso",
        "solicitações de plano de estudo"
    ],
    "saidas_esperadas": [
        "explicação conceitual",
        "exemplo simples",
        "recomendação de revisão",
        "próximo passo de estudo"
    ],
    "estado_necessario": [
        "aluno",
        "módulo atual",
        "dúvida atual",
        "tópico detectado",
        "histórico recente"
    ],
    "memorias_necessarias": [
        "módulos concluídos",
        "dificuldades recorrentes",
        "preferências de explicação"
    ],
    "ferramentas_futuras": [
        "consultar cronograma",
        "consultar progresso",
        "gerar plano de estudo"
    ],
    "regras_de_comportamento": [
        "explicar conceitos antes de código",
        "conectar respostas ao módulo do curso",
        "não avançar para implementação se faltar base conceitual"
    ],
    "criterios_de_sucesso": [
        "aluno entende o conceito principal",
        "aluno recebe próximo passo claro",
        "resposta respeita o nível do módulo"
    ],
    "criterios_de_parada": [
        "dúvida respondida",
        "plano entregue",
        "necessidade de informação externa identificada"
    ],
    "falhas_esperadas": [
        "dúvida ambígua",
        "falta de informação sobre progresso",
        "aluno pede implementação antes do módulo adequado"
    ],
    "nao_deve_fazer": [
        "inventar progresso do aluno",
        "usar ferramentas inexistentes neste módulo",
        "entrar em LangGraph antes da base conceitual"
    ]
}

pprint(ficha_tutor_langgraph)


{'criterios_de_parada': ['dúvida respondida',
                         'plano entregue',
                         'necessidade de informação externa identificada'],
 'criterios_de_sucesso': ['aluno entende o conceito principal',
                          'aluno recebe próximo passo claro',
                          'resposta respeita o nível do módulo'],
 'entradas_esperadas': ['dúvidas conceituais',
                        'pedidos de exemplo',
                        'dúvidas sobre módulos do curso',
                        'solicitações de plano de estudo'],
 'estado_necessario': ['aluno',
                       'módulo atual',
                       'dúvida atual',
                       'tópico detectado',
                       'histórico recente'],
 'falhas_esperadas': ['dúvida ambígua',
                      'falta de informação sobre progresso',
                      'aluno pede implementação antes do módulo adequado'],
 'ferramentas_futuras': ['consultar cronograma',
        

## Discussão

Essa ficha torna explícito o que o agente deve fazer e o que não deve fazer.

Ela também separa conceitos que muitas vezes ficam misturados:

- objetivo;
- entradas;
- saídas;
- estado;
- memória;
- ferramentas;
- regras;
- critérios de parada.

Essa separação será essencial quando formos implementar agentes com grafos e tools.


# 12. Entradas, saídas e responsabilidades

Um agente bem especificado deve ter fronteiras claras.

Isso significa definir:

| Elemento | Pergunta |
|---|---|
| Entrada | O que o agente recebe? |
| Saída | O que o agente entrega? |
| Responsabilidade | Pelo que o agente é responsável? |
| Limite | O que está fora do escopo? |

Sem essa definição, o agente tende a se tornar imprevisível, genérico ou difícil de testar.


In [45]:
contrato_entrada_saida = {
    "entradas": {
        "mensagem_do_aluno": "texto livre com dúvida ou solicitação",
        "modulo_atual": "identificador do módulo, quando disponível",
        "historico_recente": "mensagens anteriores relevantes"
    },
    "saidas": {
        "diagnostico": "interpretação da dúvida",
        "explicacao": "resposta conceitual ou orientação",
        "proximo_passo": "ação recomendada para o aluno"
    },
    "fora_do_escopo": [
        "executar código real neste módulo",
        "consultar banco de dados real",
        "assumir progresso sem informação"
    ]
}

pprint(contrato_entrada_saida)


{'entradas': {'historico_recente': 'mensagens anteriores relevantes',
              'mensagem_do_aluno': 'texto livre com dúvida ou solicitação',
              'modulo_atual': 'identificador do módulo, quando disponível'},
 'fora_do_escopo': ['executar código real neste módulo',
                    'consultar banco de dados real',
                    'assumir progresso sem informação'],
 'saidas': {'diagnostico': 'interpretação da dúvida',
            'explicacao': 'resposta conceitual ou orientação',
            'proximo_passo': 'ação recomendada para o aluno'}}


# 13. Estado conceitual do agente

Podemos representar o estado de forma simples usando uma classe.

Ainda não estamos usando LangGraph.

A ideia aqui é apenas tornar explícitas as informações que o agente pode precisar manter durante uma interação.


In [46]:
@dataclass
class EstadoTutorConceitual:
    aluno: str
    modulo_atual: str
    duvida_atual: str = ""
    topico_detectado: str = ""
    historico_recente: List[str] = field(default_factory=list)
    precisa_revisao: bool = False

estado = EstadoTutorConceitual(
    aluno="Renan",
    modulo_atual="A1-A3",
    duvida_atual="Qual a diferença entre chatbot e agente?"
)

estado


EstadoTutorConceitual(aluno='Renan', modulo_atual='A1-A3', duvida_atual='Qual a diferença entre chatbot e agente?', topico_detectado='', historico_recente=[], precisa_revisao=False)

## 13.1 Atualizando estado de forma conceitual

Um agente pode atualizar seu estado conforme novas mensagens chegam.

Em frameworks futuros, isso será tratado de forma mais estruturada.

Aqui, vamos fazer apenas uma simulação simples.


In [47]:
def atualizar_estado_com_duvida(estado: EstadoTutorConceitual, nova_duvida: str) -> EstadoTutorConceitual:
    estado.historico_recente.append(estado.duvida_atual)
    estado.duvida_atual = nova_duvida

    texto = nova_duvida.lower()
    if "chatbot" in texto and "agente" in texto:
        estado.topico_detectado = "diferença entre chatbot e agente"
    elif "memória" in texto or "memoria" in texto:
        estado.topico_detectado = "memória"
    elif "estado" in texto:
        estado.topico_detectado = "estado"
    else:
        estado.topico_detectado = "tópico geral"

    estado.precisa_revisao = estado.topico_detectado in ["estado", "memória"]
    return estado

estado = atualizar_estado_com_duvida(estado, "Qual a diferença entre estado e memória?")
estado


EstadoTutorConceitual(aluno='Renan', modulo_atual='A1-A3', duvida_atual='Qual a diferença entre estado e memória?', topico_detectado='memória', historico_recente=['Qual a diferença entre chatbot e agente?'], precisa_revisao=True)

# 14. Comportamentos e políticas

Além de estado, o agente precisa de regras ou políticas de comportamento.

Essas políticas podem ser descritas em linguagem natural, em tabelas ou, futuramente, em código.

Exemplo:

| Condição | Comportamento esperado |
|---|---|
| Dúvida conceitual | explicar com analogia |
| Dúvida sobre módulo futuro | contextualizar e adiar detalhes |
| Falta de informação | perguntar ou consultar ferramenta futuramente |
| Aluno confunde dois conceitos | comparar lado a lado |


In [48]:
def escolher_comportamento(estado: EstadoTutorConceitual) -> str:
    if "estado e memória" in estado.duvida_atual.lower() or "estado e memoria" in estado.duvida_atual.lower():
        return "comparar conceitos lado a lado"
    if estado.precisa_revisao:
        return "explicar conceito e recomendar revisão"
    if estado.modulo_atual == "A1-A3":
        return "manter resposta conceitual, sem implementação"
    return "responder normalmente"

comportamento = escolher_comportamento(estado)
print(comportamento)


comparar conceitos lado a lado


# 15. Falhas esperadas e limites do agente

Projetar um agente também significa prever falhas.

Exemplos:

| Falha | Estratégia segura |
|---|---|
| Dúvida ambígua | pedir esclarecimento |
| Falta de dados | não inventar; pedir ou consultar fonte |
| Pedido fora do escopo | explicar limite e redirecionar |
| Excesso de autonomia | pedir confirmação humana |
| Resposta incerta | explicitar incerteza |

Um agente confiável não é aquele que sempre responde com confiança. É aquele que sabe quando limitar sua ação.


In [50]:
falhas_e_respostas = {
    "duvida_ambigua": "pedir esclarecimento ou oferecer interpretações possíveis",
    "falta_de_dados": "não inventar dados; solicitar informação ou consultar fonte externa futuramente",
    "fora_do_escopo": "explicar limite e redirecionar ao tópico do curso",
    "acao_sensivel": "pedir confirmação humana antes de agir",
    "incerteza": "declarar incerteza e propor verificação"
}

pprint(falhas_e_respostas)


{'acao_sensivel': 'pedir confirmação humana antes de agir',
 'duvida_ambigua': 'pedir esclarecimento ou oferecer interpretações possíveis',
 'falta_de_dados': 'não inventar dados; solicitar informação ou consultar '
                   'fonte externa futuramente',
 'fora_do_escopo': 'explicar limite e redirecionar ao tópico do curso',
 'incerteza': 'declarar incerteza e propor verificação'}


# 16. De prompt para arquitetura

Um erro comum é tentar colocar toda a ``inteligência'' do agente no prompt.

O prompt é importante, mas ele deve ser visto como uma camada dentro de uma arquitetura maior.

| No prompt aparece como... | Na arquitetura vira... |
|---|---|
| "Você é um tutor" | papel do agente |
| "Seu objetivo é ajudar alunos" | objetivo |
| "Antes de responder, identifique o módulo" | etapa de interpretação |
| "Não invente progresso" | restrição / guardrail |
| "Use dados do aluno quando disponíveis" | memória ou consulta externa |
| "Sugira próximo passo" | política de resposta |
| "Responda em formato X" | contrato de saída |

Essa transição é o primeiro passo para sair de prompts soltos e chegar a agentes estruturados.


## 16.1 Prompt não substitui estado

Um prompt pode dizer:

```text
Lembre-se do módulo atual do aluno.
```

Mas, em um sistema real, essa informação precisa estar em algum lugar:

- no histórico da conversa;
- em um objeto de estado;
- em um banco de dados;
- em uma memória externa;
- em uma estrutura controlada pelo framework.

Por isso, prompt e arquitetura devem trabalhar juntos.


## 16.2 Prompt não substitui ferramentas

Um prompt pode dizer:

```text
Consulte o progresso do aluno antes de recomendar o próximo módulo.
```

Mas, para consultar progresso, o agente precisa de acesso a algum recurso externo:

- banco de dados;
- API;
- arquivo;
- função;
- serviço computacional.

Será estudado futuramente no módulo de Ferramentas.


# 17. Ponte para os próximos notebooks

Este notebook não implementa LangGraph ainda.

Ele prepara a linguagem conceitual que será usada depois.

| Conceito deste notebook | Implementação futura |
|---|---|
| Percepção | mensagens de entrada e observações |
| Estado | estado explícito do fluxo |
| Decisão | nó com lógica, classificador ou LLM |
| Ação | resposta, função ou tool |
| Transição | passagem de uma etapa para outra |
| Loop | repetição controlada de decisão e ação |
| Memória | histórico, store ou persistência |
| Planejamento | decomposição em etapas ou nós |
| Ferramentas | APIs, bancos, funções e serviços externos |
| Agente completo | arquitetura executável |

Nos próximos notebooks, esses conceitos deixam de ser apenas ideias e passam a ser implementados.


## 17.1 De-para com o cronograma do curso

| Bloco do curso | Foco |
|---|---|
| A1 | Conceito de agente, percepção, decisão, ação e tipos de agentes |
| A2 | Arquitetura de agentes com LLMs: modelo, prompt, memória, planejamento e ferramentas |
| A3 | Especificação de agentes: objetivos, estados, entradas, saídas e responsabilidades |
| A4 | Fundamentos práticos de LangGraph: estado, nós e transições |
| A5 | Workflows completos com LangGraph: condições, loops e estado compartilhado |
| A6 | Tools e agentes ReAct: chamada de ferramentas e interpretação de resultados |

Este notebook cobre A1, A2 e A3 como preparação conceitual para a parte prática.


# 18. Exercícios

Os exercícios a seguir são conceituais. O objetivo é praticar especificação e modelagem mental antes da implementação.


## Exercício 1 — Classificando agentes

Para cada caso abaixo, diga qual tipo de agente parece mais adequado:

1. Um sistema que responde dúvidas frequentes sobre o curso.
2. Um tutor que lembra quais módulos o aluno concluiu.
3. Um sistema que monta plano de estudo de sete dias.
4. Um agente que consulta uma API para verificar materiais disponíveis.
5. Um conjunto de agentes em que um explica, outro avalia e outro revisa.

Não há necessariamente uma única resposta correta. O objetivo é justificar a classificação.

## Exercício 2 — Programa, chatbot ou agente?

Classifique cada sistema como programa tradicional, chatbot simples ou agente.

1. Um script que calcula a média das notas.
2. Um assistente que responde "o que é LangGraph?".
3. Um tutor que detecta a dificuldade do aluno e recomenda revisão.
4. Um sistema que consulta progresso e gera plano de estudo.
5. Um fluxo que decide entre explicar, perguntar ou chamar uma ferramenta.

Justifique cada resposta.


## Exercício 3 — Percepção, decisão e ação

Escolha uma dúvida de aluno, por exemplo:

```text
"Não entendi a diferença entre estado e memória."
```

Preencha:

```text
Percepção:
Decisão:
Ação:
Estado necessário:
Próximo passo:
```


## Exercício 4 — Especificar um agente

Projete um agente para ajudar alunos a estudar o módulo A4.

Preencha:

```text
Nome do agente:
Objetivo:
Entradas:
Saídas:
Estado necessário:
Memória necessária:
Ações possíveis:
Ferramentas futuras:
Critérios de parada:
O que não deve fazer:
```


## Exercício 5 — Prompt como especificação

Transforme a especificação do exercício anterior em um prompt inicial.

O prompt deve conter:

1. papel do agente;
2. objetivo;
3. contexto do curso;
4. restrições;
5. critérios de decisão;
6. formato de saída.

Lembre-se: o prompt é uma especificação comportamental, não a arquitetura completa.


## Exercício 6 — Few-shot

Crie dois exemplos de entrada e saída esperada para o agente do exercício anterior.

Exemplo de formato:

```text
Entrada:
"Não entendi o que é um nó."

Saída esperada:
- Identificar que a dúvida é sobre A4.
- Explicar nó como unidade de processamento.
- Dar exemplo simples.
- Sugerir próximo passo.
```


## Exercício 7 — Critérios de decisão

Crie uma tabela com pelo menos quatro critérios de decisão para o agente tutor.

Exemplo:

| Condição | Ação |
|---|---|
| aluno pede código antes da teoria | explicar conceito primeiro |
| aluno confunde dois conceitos | comparar lado a lado |
| falta informação sobre módulo atual | perguntar |
| precisa de dado externo | usar ferramenta futuramente |


# 19. Síntese

Este notebook apresentou que agentes inteligentes são sistemas que percebem, decidem e agem em um ambiente.

Vimos que agentes com LLMs não são apenas chatbots. Eles podem combinar:

- modelo de linguagem;
- prompt;
- estado;
- memória;
- raciocínio estruturado;
- planejamento;
- ferramentas;
- critérios de decisão;
- critérios de parada.

A principal mensagem é:

> Antes de implementar um agente, especifique o agente.

Essa especificação prepara a transição para frameworks e arquiteturas mais concretas.


## 19.1 Checklist de entendimento

Antes de avançar, verifique se você consegue responder:

- O que diferencia um agente de um programa tradicional?
- O que diferencia um chatbot simples de um agente?
- O que são percepção, decisão e ação?
- O que é estado em um agente?
- Qual é a diferença entre estado e memória?
- Qual é o papel da LLM em um agente?
- Por que prompt não é o agente inteiro?
- Para que serve few-shot prompting?
- Por que preferimos raciocínio estruturado observável em vez de pedir cadeia completa de pensamento?
- Como especificar objetivo, entradas, saídas e responsabilidades de um agente?


## 19.2 Próximos passos

Nos próximos notebooks veremos:

- conceitos como estado, nós e transições aparecem em grafos de execução;
- construiremos workflows com condições e loops;
- adicionaremos LLMs aos fluxos;
- conectaremos ferramentas externas;
- entenderemos agentes ReAct e o ciclo de chamada de tools.


# 20. Referências e leituras complementares

Sugestões de leitura para aprofundamento:

- Russell, S. e Norvig, P. *Artificial Intelligence: A Modern Approach* — capítulos sobre agentes inteligentes.
- Wooldridge, M. *An Introduction to MultiAgent Systems* — fundamentos de sistemas multiagentes.
- Yao et al. *ReAct: Synergizing Reasoning and Acting in Language Models* — base conceitual para agentes que combinam raciocínio e ação.
- Documentação do LangChain — visão geral de agentes, tools e modelos de linguagem.
- Documentação do LangGraph — workflows, agentes, estado e controle de execução.

Neste notebook, essas referências aparecem apenas como contexto conceitual. A implementação será tratada nos próximos módulos.
